# 🚗 Automotive Medallion Pipeline — Databricks Edition

End-to-end data engineering pipeline on Databricks Free Edition, processing a
Kaggle car-specs dataset through the **Bronze → Silver → Gold** medallion architecture.

**Stack:** PySpark · Delta Lake · Spark SQL · Unity Catalog · Databricks AI/BI Dashboards

**Pipeline:**
- **Bronze** — raw ingestion, all columns as strings
- **Silver** — PySpark cleaning, type casting, business classifications, data-quality flags
- **Gold** — aggregated Spark SQL tables ready for analysis and dashboards

This is the Databricks counterpart to a parallel Postgres + dbt build of the same domain,
demonstrating the same logic across two distinct stacks.

## 🥉 Bronze Layer — Raw Ingestion

The raw CSV was uploaded via Databricks *Create table from file upload* into
`workspace.layer_bronze.raw_cars`. All 11 columns are kept as raw strings —
no cleaning happens here, preserving the source data faithfully.


In [0]:
# Read the Bronze Delta table into a Spark DataFrame
df_bronze = spark.table("workspace.layer_bronze.raw_cars")

# Inspect the raw data
print(f"Total records: {df_bronze.count()}")
df_bronze.printSchema()
df_bronze.display()

Total records: 1218
root
 |-- Company Names: string (nullable = true)
 |-- Cars Names: string (nullable = true)
 |-- Engines: string (nullable = true)
 |-- CC/Battery Capacity: string (nullable = true)
 |-- HorsePower: string (nullable = true)
 |-- Total Speed: string (nullable = true)
 |-- Performance(0 - 100 )KM/H: string (nullable = true)
 |-- Cars Prices: string (nullable = true)
 |-- Fuel Types: string (nullable = true)
 |-- Seats: string (nullable = true)
 |-- Torque: string (nullable = true)



Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
FERRARI,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
ROLLS ROYCE,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
Ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
MERCEDES,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
AUDI,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm
BMW,Mclaren 720s,V8,"3,994 cc",710 hp,341 km/h,2.9 sec,"$499,000",Petrol,2,770 Nm
ASTON MARTIN,VANTAGE F1,V8,"3,982 cc",656 hp,314 km/h,3.6 sec,"$193,440",Petrol,2,685 Nm
BENTLEY,Continental GT Azure,V8,"3,996 cc",550 hp,318 km/h,4.0 sec,"$311,000",Petrol,4,900 Nm
LAMBORGHINI,VENENO ROADSTER,V12,"6,498 cc",750 hp,356 km/h,2.9 sec,"$4,500,000",Petrol,2,690 Nm
FERRARI,F8 TRIBUTO,V8,"3,900 cc",710 hp,340 km/h,2.9 sec,"$280,000",Petrol,2,770 Nm


## 🥈 Silver Layer — Cleaning & Enrichment

Transforms raw strings into typed, validated fields:
- **Numeric extraction** via regex helpers (`963 hp` → `963`), using `try_cast`
  to return NULL on malformed values instead of failing.
- **fuel_category** — thermal fuels take priority over hybrid/electric
  (a diesel-charged hybrid counts as Diesel). Handles source typos (`hyrbrid`).
- **engine_cc / battery_capacity_kwh** — split from a shared column by fuel type.
- **vehicle_class** — luxury brands are always `light`; large thermal engines → `heavy` (trucks).
- **price_segment** — Economy / Premium / Luxury for cars; `Commercial` for trucks.
- **is_showroom_model** — a data-quality flag marking non-showroom vehicles
  (race cars, prototypes) for exclusion downstream, without deleting them from Silver.


In [0]:
from pyspark.sql import functions as F

# Helper: extract the first integer from a messy string.
# .try_cast() returns NULL instead of failing when the value is empty/malformed.
def extract_int(col_name):
    cleaned = F.regexp_replace(F.col(col_name), ",", "")
    extracted = F.regexp_extract(cleaned, r"[0-9]+", 0)
    return extracted.try_cast("int")

# Helper: extract the first decimal number (for values like "2.5 sec")
def extract_decimal(col_name):
    cleaned = F.regexp_replace(F.col(col_name), ",", "")
    extracted = F.regexp_extract(cleaned, r"[0-9]+\.?[0-9]*", 0)
    return extracted.try_cast("double")

In [0]:
df_silver = (
    df_bronze
    # --- Text normalization ---
    .withColumn("company_name", F.upper(F.trim(F.col("Company Names"))))
    .withColumn("car_name", F.trim(F.col("Cars Names")))
    .withColumn("engine_config", F.trim(F.col("Engines")))
    # --- Numeric extraction (using our helpers) ---
    .withColumn("horsepower", extract_int("HorsePower"))
    .withColumn("top_speed_kmh", extract_int("Total Speed"))
    .withColumn("acceleration_0_100", extract_decimal("Performance(0 - 100 )KM/H"))
    .withColumn("price_usd", extract_int("Cars Prices"))
    .withColumn("seats", extract_int("Seats"))
)

# --- Fix Jaguar Land Rover -> Jaguar (dataset has only Jaguar models) ---
df_silver = df_silver.withColumn(
    "company_name",
    F.when(F.col("company_name") == "JAGUAR LAND ROVER", "JAGUAR")
     .otherwise(F.col("company_name"))
)

# --- Flag non-showroom models (race cars, concepts, limited prototypes) ---
# These are real data points we keep in Silver, but exclude from commercial
# analysis in Gold. Centralizing the rule here avoids repeating filters everywhere.
special_models = ["Race Car", "XL1"]
special_pattern = "|".join(special_models)

df_silver = df_silver.withColumn(
    "is_showroom_model",
    ~F.col("car_name").rlike(special_pattern)
)

# --- fuel_category: thermal takes priority over hybrid/electric ---
fuel_raw = F.lower(F.col("Fuel Types"))
df_silver = df_silver.withColumn(
    "fuel_category",
    F.when(fuel_raw.rlike("diesel"), "Diesel")
     .when(fuel_raw.rlike("petrol|gas"), "Petrol")
     .when(fuel_raw.rlike("hydrogen"), "Hydrogen")
     .when(fuel_raw.rlike("hybrid|hyrbrid|hyrbid"), "Hybrid")
     .when(fuel_raw.rlike("electric"), "Electric")
     .otherwise(None)
)

# --- Split CC/Battery into engine_cc (thermal) and battery_capacity_kwh (electric) ---
cc_value = extract_int("CC/Battery Capacity")
df_silver = (
    df_silver
    .withColumn("engine_cc",
        F.when(F.col("fuel_category") != "Electric", cc_value).otherwise(None))
    .withColumn("battery_capacity_kwh",
        F.when(F.col("fuel_category") == "Electric", cc_value).otherwise(None))
)

# --- vehicle_class: luxury brands always 'light'; else heavy if big thermal engine ---
luxury_brands = ["ASTON MARTIN", "BENTLEY", "BUGATTI", "FERRARI",
                 "LAMBORGHINI", "ROLLS ROYCE", "PORSCHE"]
df_silver = df_silver.withColumn(
    "vehicle_class",
    F.when(F.col("company_name").isin(luxury_brands), "light")
     .when((F.col("engine_cc") > 7000) & (F.col("fuel_category") != "Electric"), "heavy")
     .otherwise("light")
)

# --- price_segment: heavy vehicles (trucks) get 'Commercial'; cars get price tiers ---
df_silver = df_silver.withColumn(
    "price_segment",
    F.when(F.col("vehicle_class") == "heavy", "Commercial")
     .when(F.col("price_usd") < 40000, "Economy")
     .when(F.col("price_usd") < 100000, "Premium")
     .otherwise("Luxury")
)

# --- Final select: keep only clean columns, drop rows without a price ---
df_silver = df_silver.select(
    "company_name", "car_name", "engine_config", "engine_cc",
    "battery_capacity_kwh", "horsepower", "top_speed_kmh",
    "acceleration_0_100", "price_usd", "seats",
    "fuel_category", "vehicle_class", "price_segment",
    "is_showroom_model"
).filter(F.col("price_usd").isNotNull())

df_silver.display()

company_name,car_name,engine_config,engine_cc,battery_capacity_kwh,horsepower,top_speed_kmh,acceleration_0_100,price_usd,seats,fuel_category,vehicle_class,price_segment,is_showroom_model
FERRARI,SF90 STRADALE,V8,3990,null,963,340,2.5,1100000,2,Hybrid,light,Luxury,true
ROLLS ROYCE,PHANTOM,V12,6749,null,563,250,5.3,460000,5,Petrol,light,Luxury,true
FORD,KA+,1.2L Petrol,1200,null,70,165,10.5,12000,5,Petrol,light,Economy,true
MERCEDES,GT 63 S,V8,3982,null,630,250,3.2,161000,4,Petrol,light,Luxury,true
AUDI,AUDI R8 Gt,V10,5204,null,602,320,3.6,253290,2,Petrol,light,Luxury,true
BMW,Mclaren 720s,V8,3994,null,710,341,2.9,499000,2,Petrol,light,Luxury,true
ASTON MARTIN,VANTAGE F1,V8,3982,null,656,314,3.6,193440,2,Petrol,light,Luxury,true
BENTLEY,Continental GT Azure,V8,3996,null,550,318,4.0,311000,4,Petrol,light,Luxury,true
LAMBORGHINI,VENENO ROADSTER,V12,6498,null,750,356,2.9,4500000,2,Petrol,light,Luxury,true
FERRARI,F8 TRIBUTO,V8,3900,null,710,340,2.9,280000,2,Petrol,light,Luxury,true


In [0]:
# Create the silver schema if it doesn't exist yet
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

# Persist the Silver DataFrame as a managed Delta table.
# overwriteSchema=true allows changing the table structure (we added is_showroom_model).
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.clean_cars")
)

print("Silver table written: workspace.silver.clean_cars")
print(f"Rows: {spark.table('workspace.silver.clean_cars').count()}")

Silver table written: workspace.silver.clean_cars
Rows: 1217


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

DataFrame[]

## 🥇 Gold Layer — Business Aggregations

Analysis-ready tables built with **Spark SQL** (the right tool for aggregations,
as PySpark is the right tool for the complex row-level logic in Silver).

All Gold tables filter on `is_showroom_model = true` — the single, centralized
data-quality gate defined in Silver.

In [0]:
# Create the gold schema
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

DataFrame[]

In [0]:
# --- Gold table 1: brand_metrics ---
# Aggregated metrics per brand.
# Excludes heavy vehicles (trucks) AND race cars — only street-legal cars.
spark.sql("""
CREATE OR REPLACE TABLE workspace.gold.brand_metrics AS
SELECT
    company_name,
    COUNT(*)                              AS total_models,
    ROUND(AVG(price_usd))                 AS avg_price_usd,
    MAX(price_usd)                        AS max_price_usd,
    MIN(price_usd)                        AS min_price_usd,
    MAX(price_usd) - MIN(price_usd)       AS price_range_usd,
    ROUND(AVG(horsepower))                AS avg_horsepower,
    ROUND(AVG(top_speed_kmh))             AS avg_top_speed,
    ROUND(AVG(acceleration_0_100), 2)     AS avg_acceleration,
    ROUND(AVG(price_usd) / NULLIF(AVG(horsepower), 0))    AS avg_price_per_hp,
    ROUND(AVG(price_usd) / NULLIF(AVG(top_speed_kmh), 0)) AS avg_price_per_kmh
FROM workspace.silver.clean_cars
WHERE vehicle_class = 'light'
  AND is_showroom_model = true
GROUP BY company_name
ORDER BY avg_price_usd DESC
""")

spark.table("workspace.gold.brand_metrics").display()

company_name,total_models,avg_price_usd,max_price_usd,min_price_usd,price_range_usd,avg_horsepower,avg_top_speed,avg_acceleration,avg_price_per_hp,avg_price_per_kmh
BUGATTI,10,5870000.0,18000000,3000000,15000000,1565.0,420.0,2.4,3751.0,13976.0
ASTON MARTIN,11,752949.0,3200000,142000,3058000,701.0,330.0,3.53,1074.0,2282.0
LAMBORGHINI,24,650083.0,4500000,211000,4289000,692.0,334.0,3.04,940.0,1946.0
FERRARI,9,512222.0,1700000,210000,1490000,710.0,331.0,3.03,722.0,1547.0
ROLLS ROYCE,30,390400.0,515000,320000,195000,584.0,250.0,4.91,668.0,1562.0
BENTLEY,1,311000.0,311000,311000,0,550.0,318.0,4.0,565.0,978.0
PORSCHE,96,138390.0,750000,58000,692000,463.0,278.0,4.25,299.0,497.0
TESLA,10,87032.0,200000,40240,159760,693.0,262.0,3.51,126.0,333.0
MERCEDES,21,84190.0,200000,46000,154000,433.0,250.0,4.76,194.0,337.0
AUDI,21,82871.0,253290,35000,218290,393.0,260.0,5.16,211.0,318.0


In [0]:
# --- Gold table 2: fuel_by_segment ---
# Metrics per fuel category WITHIN each price segment.
# Grouping by both dimensions avoids averaging a Ferrari with a Mazda.
spark.sql("""
CREATE OR REPLACE TABLE workspace.gold.fuel_by_segment AS
SELECT
    price_segment,
    fuel_category,
    COUNT(*)                              AS total_models,
    ROUND(AVG(price_usd))                 AS avg_price_usd,
    ROUND(AVG(horsepower))                AS avg_horsepower,
    ROUND(AVG(top_speed_kmh))             AS avg_top_speed,
    ROUND(AVG(acceleration_0_100), 2)     AS avg_acceleration
FROM workspace.silver.clean_cars
WHERE vehicle_class = 'light'
  AND is_showroom_model = true
  AND fuel_category IS NOT NULL
GROUP BY price_segment, fuel_category
ORDER BY
    CASE price_segment
        WHEN 'Economy' THEN 1
        WHEN 'Premium' THEN 2
        WHEN 'Luxury'  THEN 3
    END,
    avg_price_usd DESC
""")

spark.table("workspace.gold.fuel_by_segment").display()

price_segment,fuel_category,total_models,avg_price_usd,avg_horsepower,avg_top_speed,avg_acceleration
Economy,Hydrogen,1,38000.0,161.0,180.0,7.8
Economy,Hybrid,27,30911.0,175.0,185.0,9.03
Economy,Electric,21,27287.0,147.0,141.0,8.19
Economy,Diesel,79,25958.0,164.0,174.0,11.65
Economy,Petrol,423,24676.0,158.0,193.0,9.51
Premium,Electric,58,63676.0,369.0,196.0,5.9
Premium,Petrol,311,59594.0,336.0,229.0,6.14
Premium,Diesel,42,53060.0,263.0,181.0,9.11
Premium,Hydrogen,2,50950.0,152.0,180.0,9.1
Premium,Hybrid,44,50258.0,292.0,210.0,6.85


In [0]:
# --- Gold tables: brand metrics split by each price tier ---
# One table per segment keeps each chart on a coherent price scale.
# A brand can appear in multiple tables if it sells cars across tiers
# (e.g. Porsche has both Premium and Luxury models) — this is intended.

def build_segment_table(table_name, segment):
    spark.sql(f"""
    CREATE OR REPLACE TABLE workspace.gold.{table_name} AS
    SELECT
        company_name,
        '{segment}'                       AS price_segment,
        COUNT(*)                          AS total_models,
        ROUND(AVG(price_usd))             AS avg_price_usd,
        ROUND(AVG(horsepower))            AS avg_horsepower,
        ROUND(AVG(top_speed_kmh))         AS avg_top_speed
    FROM workspace.silver.clean_cars
    WHERE vehicle_class = 'light'
      AND is_showroom_model = true
      AND price_segment = '{segment}'
    GROUP BY company_name
    ORDER BY avg_price_usd DESC
    """)

# Build one table per segment
build_segment_table("brand_metrics_economy", "Economy")
build_segment_table("brand_metrics_premium", "Premium")
build_segment_table("brand_metrics_luxury",  "Luxury")

print("Created brand_metrics_economy, brand_metrics_premium, brand_metrics_luxury")
print("\n--- Economy ---")
spark.table("workspace.gold.brand_metrics_economy").display()
print("\n--- Premium ---")
spark.table("workspace.gold.brand_metrics_premium").display()
print("\n--- Luxury ---")
spark.table("workspace.gold.brand_metrics_luxury").display()

Created brand_metrics_economy, brand_metrics_premium, brand_metrics_luxury

--- Economy ---


company_name,price_segment,total_models,avg_price_usd,avg_horsepower,avg_top_speed
AUDI,Economy,2,37000.0,248.0,235.0
CADILLAC,Economy,3,36693.0,236.0,237.0
GMC,Economy,12,35208.0,218.0,189.0
JEEP,Economy,5,34320.0,225.0,190.0
BMW,Economy,21,32905.0,142.0,211.0
ACURA,Economy,5,30600.0,202.0,202.0
PEUGEOT,Economy,37,30176.0,143.0,194.0
TOYOTA,Economy,26,28988.0,196.0,195.0
CHEVROLET,Economy,26,28104.0,204.0,193.0
KIA,Economy,44,28034.0,172.0,196.0



--- Premium ---


company_name,price_segment,total_models,avg_price_usd,avg_horsepower,avg_top_speed
PORSCHE,Premium,34,81759.0,359.0,254.0
JAGUAR,Premium,35,67143.0,326.0,242.0
GMC,Premium,41,64905.0,396.0,191.0
TESLA,Premium,7,64049.0,515.0,233.0
AUDI,Premium,14,63929.0,343.0,245.0
VOLVO,Premium,4,62500.0,242.0,188.0
MERCEDES,Premium,14,61643.0,353.0,250.0
CHEVROLET,Premium,32,58661.0,385.0,206.0
BMW,Premium,13,58615.0,346.0,258.0
CADILLAC,Premium,15,58594.0,370.0,228.0



--- Luxury ---


company_name,price_segment,total_models,avg_price_usd,avg_horsepower,avg_top_speed
BUGATTI,Luxury,10,5870000.0,1565.0,420.0
ASTON MARTIN,Luxury,11,752949.0,701.0,330.0
LAMBORGHINI,Luxury,24,650083.0,692.0,334.0
FERRARI,Luxury,9,512222.0,710.0,331.0
FORD,Luxury,1,500000.0,660.0,348.0
NISSAN,Luxury,9,413111.0,574.0,313.0
ROLLS ROYCE,Luxury,30,390400.0,584.0,250.0
BENTLEY,Luxury,1,311000.0,550.0,318.0
BMW,Luxury,7,186714.0,538.0,297.0
TOYOTA,Luxury,1,170000.0,406.0,200.0


In [0]:
# --- Gold: price efficiency & power sweet-spot (PETROL ONLY) ---
# Two distinct horsepower filters, different in intent:
#   - horsepower <= 2000  -> VALIDITY filter: excludes impossible values
#     (Nissan Urvan's 2488 hp is a data-entry error, likely 2488cc displacement).
#   - horsepower >= 50    -> SCOPE filter: excludes valid-but-off-target cars
#     (microcars & vintage models like Tata Nano, Karmann Ghia) that don't
#     represent the modern car market we're analyzing.

HP_MIN = 50      # scope: modern market cars
HP_MAX = 2000    # validity: physically possible

# Analysis 1: average price-per-hp by segment, petrol only.
spark.sql(f"""
CREATE OR REPLACE TABLE workspace.gold.price_per_hp_petrol AS
SELECT
    price_segment,
    COUNT(*)                                          AS total_models,
    ROUND(AVG(price_usd / NULLIF(horsepower, 0)))     AS avg_price_per_hp,
    ROUND(AVG(price_usd))                             AS avg_price_usd,
    ROUND(AVG(horsepower))                            AS avg_horsepower
FROM workspace.silver.clean_cars
WHERE vehicle_class = 'light'
  AND is_showroom_model = true
  AND fuel_category = 'Petrol'
  AND horsepower BETWEEN {HP_MIN} AND {HP_MAX}
  AND price_segment IS NOT NULL
GROUP BY price_segment
ORDER BY
    CASE price_segment
        WHEN 'Economy' THEN 1
        WHEN 'Premium' THEN 2
        WHEN 'Luxury'  THEN 3
    END
""")

print("--- Price per HP (Petrol) ---")
spark.table("workspace.gold.price_per_hp_petrol").display()

# Analysis 2: power sweet-spot by segment, petrol only.
spark.sql(f"""
CREATE OR REPLACE TABLE workspace.gold.power_sweet_spot AS
SELECT
    price_segment,
    COUNT(*)                          AS total_models,
    ROUND(AVG(horsepower))            AS avg_horsepower,
    ROUND(STDDEV(horsepower))         AS stddev_horsepower,
    MIN(horsepower)                   AS min_horsepower,
    MAX(horsepower)                   AS max_horsepower
FROM workspace.silver.clean_cars
WHERE vehicle_class = 'light'
  AND is_showroom_model = true
  AND fuel_category = 'Petrol'
  AND horsepower BETWEEN {HP_MIN} AND {HP_MAX}
  AND price_segment IS NOT NULL
GROUP BY price_segment
ORDER BY
    CASE price_segment
        WHEN 'Economy' THEN 1
        WHEN 'Premium' THEN 2
        WHEN 'Luxury'  THEN 3
    END
""")

print("--- Power Sweet Spot (Petrol) ---")
spark.table("workspace.gold.power_sweet_spot").display()

--- Price per HP (Petrol) ---


price_segment,total_models,avg_price_per_hp,avg_price_usd,avg_horsepower
Economy,418,163.0,24834.0,159.0
Premium,311,182.0,59594.0,336.0
Luxury,164,714.0,651685.0,634.0


--- Power Sweet Spot (Petrol) ---


price_segment,total_models,avg_horsepower,stddev_horsepower,min_horsepower,max_horsepower
Economy,418,159.0,61.0,52,381
Premium,311,336.0,85.0,150,760
Luxury,164,634.0,259.0,325,1850
